In [1]:
import pandas as pd
import numpy as np

In [2]:
# =========================================================
# UNHCR DEMOGRAPHICS CLEANING (Refugee population by age/sex)
# Output: clean_demographics_aggregated.csv
# Notes:
# - Keeps years >= 2010
# - Drops unknown origin code (UKN)
# - Converts numeric columns safely
# - Treats missing demographic counts as 0 AFTER conversion (document this in methodology)
# =========================================================

print("🚀 Starting UNHCR demographics cleaning & aggregation...")

DEMOGRAPHICS_IN = "./data/demographics.csv"
DEMOGRAPHICS_OUT = "./data/clean_demographics_aggregated.csv"

# Load
df = pd.read_csv(DEMOGRAPHICS_IN, low_memory=False)

# Clean year
df = df[df["Year"] != "#date+year"]
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df.dropna(subset=["Year"])
df["Year"] = df["Year"].astype(int)

# Filter years (2010+)
df = df[df["Year"] >= 2010]

# Drop unknown nationality (UKN) origin
df = df[df["Country of Origin Code"] != "UKN"]

# Numeric columns to clean
numeric_cols = [
    "Female 0-4", "Male 0-4",
    "Female 5-11", "Male 5-11",
    "Female 12-17", "Male 12-17",
    "Female 18-59", "Male 18-59",
    "Female 60 or more", "Male 60 or more"
]

# Convert to numeric, then fill missing with 0 (counts)
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols] = df[numeric_cols].fillna(0)

# Aggregate (sum) by year + origin + asylum
group_cols = [
    "Year",
    "Country of Origin Code", "Country of Origin Name",
    "Country of Asylum Code", "Country of Asylum Name"
]

df_agg = (
    df.groupby(group_cols, as_index=False)[numeric_cols]
      .sum()
)

df_agg.to_csv(DEMOGRAPHICS_OUT, index=False)
print(f"✅ UNHCR Done! Saved: {DEMOGRAPHICS_OUT} | shape={df_agg.shape}")


🚀 Starting UNHCR demographics cleaning & aggregation...
✅ UNHCR Done! Saved: ./data/clean_demographics_aggregated.csv | shape=(76365, 15)


In [4]:

# =========================================================
# WORLD BANK EDUCATION CLEANING (Education Statistics)
# Input: education.csv (your provided file)
# Output: clean_education.csv
#
# Goal:
# - Keep selected countries (hosts + context)
# - Keep indicators relevant to primary/secondary enrollment
# - Keep years 2010-2024 only
# - Drop rows that have NO data across the selected years
#
# IMPORTANT (methodology):
# - We do NOT generate random values for missing data.
# - Missing stays missing, and the visualization must explicitly mark it.
# =========================================================

print("\n🚀 Starting World Bank education cleaning...")

EDU_IN = "./data/WDICSV.csv"          # your uploaded World Bank file
EDU_OUT = "./data/clean_educations.csv"

# Countries (host + context)
target_countries = [
    "TUR", "LBN", "JOR", "PAK", "UGA", "DEU", "SDN", "ETH",  # Host countries
    "WLD",                                                  # World
    "SYR", "AFG"                                            # Origin context (optional)
]

# Indicators (prefer Net; keep fallback options)
# Primary:
# - SE.PRM.NENR: School enrollment, primary (% net)
# - SE.PRM.TENR: Adjusted net enrollment rate, primary (%)
# Secondary:
# - SE.SEC.NENR: School enrollment, secondary (% net)
# - SE.SEC.ENRR: School enrollment, secondary (% gross) fallback
# - UIS.NERA.2  : Lower secondary net enrollment (UIS) optional fallback
target_indicators = [
    "SE.PRM.NENR",
    "SE.PRM.TENR",
    "SE.SEC.NENR",
    "SE.SEC.ENRR",
    "UIS.NERA.2"
]

# Years 2010-2024
years_to_keep = [str(y) for y in range(2010, 2025)]

try:
    edu = pd.read_csv(EDU_IN, low_memory=False)
    print(f"   > Original education shape: {edu.shape}")

    # Basic column sanity check
    required_cols = {"Country Code", "Indicator Code", "Country Name", "Indicator Name"}
    missing_cols = required_cols - set(edu.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns in education file: {missing_cols}")

    # Filter countries + indicators
    edu = edu[edu["Country Code"].isin(target_countries)]
    edu = edu[edu["Indicator Code"].isin(target_indicators)]

    # Keep metadata + available year columns
    cols_metadata = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
    available_years = [y for y in years_to_keep if y in edu.columns]

    if not available_years:
        raise ValueError("No requested year columns (2010-2024) found in the education file.")

    final_cols = cols_metadata + available_years
    edu_clean = edu[final_cols].copy()

    # Convert year columns to numeric (keep NaN if missing)
    for y in available_years:
        edu_clean[y] = pd.to_numeric(edu_clean[y], errors="coerce")

    # Drop rows with ALL missing across selected years
    edu_clean = edu_clean.dropna(subset=available_years, how="all")

    # Optional: sort for readability
    edu_clean = edu_clean.sort_values(["Country Code", "Indicator Code"]).reset_index(drop=True)

    edu_clean.to_csv(EDU_OUT, index=False)

    print(f"✅ Education Done! Saved: {EDU_OUT} | shape={edu_clean.shape}")
    print(f"   > Indicators kept: {sorted(edu_clean['Indicator Code'].unique().tolist())}")
    print(f"   > Countries kept: {sorted(edu_clean['Country Code'].unique().tolist())}")

except FileNotFoundError:
    print(f"❌ Error: '{EDU_IN}' not found.")
except Exception as e:
    print(f"❌ Error while cleaning education data: {e}")


🚀 Starting World Bank education cleaning...
   > Original education shape: (402458, 69)
✅ Education Done! Saved: ./data/clean_educations.csv | shape=(37, 19)
   > Indicators kept: ['SE.PRM.NENR', 'SE.PRM.TENR', 'SE.SEC.ENRR', 'SE.SEC.NENR']
   > Countries kept: ['AFG', 'DEU', 'ETH', 'JOR', 'LBN', 'PAK', 'SDN', 'SYR', 'TUR', 'UGA', 'WLD']


In [9]:

# =========================================================
# UNESCO SDG4 EDUCATION CLEANING (UIS SDG4 Bulk Data)
# Input:
#   - SDG_DATA_NATIONAL.csv  (long format: country-year-indicator-value)
#   - SDG_COUNTRY.csv        (country names)
# Output:
#   - clean_education.csv    (wide format like WDI: year columns)
#
# Goal:
# - Keep selected countries (hosts + context)
# - Build indicators needed for our website dumbbell (primary vs secondary)
# - Fix missing secondary using fallback logic:
#     prefer 2T3, else avg(2,3), else use only2 or only3
# - Keep years 2010-2024
# - Add extra indicators for future chart ideas (gender gap, completion, PTR, raw OOS)
#
# IMPORTANT (methodology):
# - We do NOT invent random values.
# - We only transform official indicators:
#     In-school proxy (%) = 100 - Out-of-school rate (%)
# - Fallback is explicit via a "source flag" indicator.
# =========================================================

print("\n🚀 Starting UNESCO SDG4 education cleaning...")

SDG_IN = "./data/SDG_DATA_NATIONAL.csv"
COUNTRY_IN = "./data/SDG_COUNTRY.csv"
EDU_OUT = "./data/clean_education.csv"

# Countries (host + context) - same style as your WDI script
target_countries = [
    "TUR", "LBN", "IRN", "KEN", "PAK", "UGA", "DEU", "ETH",  # Host countries
    "WLD",                                                  # World
    "SYR", "AFG","SDN","JOR"                                            # Origin context (optional)
]

# Years 2010-2024
years_to_keep = list(range(2010, 2025))
years_str = [str(y) for y in years_to_keep]

# --- UNESCO SDG4 Indicator IDs (raw) ---
# Out-of-school (both sexes)
OOS_PRIMARY = "ROFST.1.CP"
OOS_LOWER   = "ROFST.2.CP"
OOS_UPPER   = "ROFST.3.CP"
OOS_2T3     = "ROFST.2T3.CP"

# Out-of-school by sex (optional but useful)
OOS_PRIMARY_F = "ROFST.1.F.CP"
OOS_PRIMARY_M = "ROFST.1.M.CP"
OOS_LOWER_F   = "ROFST.2.F.CP"
OOS_LOWER_M   = "ROFST.2.M.CP"
OOS_UPPER_F   = "ROFST.3.F.CP"
OOS_UPPER_M   = "ROFST.3.M.CP"
OOS_2T3_F     = "ROFST.2T3.F.CP"
OOS_2T3_M     = "ROFST.2T3.M.CP"

# Completion rates (often present in SDG4 bulk)
CR_PRIMARY = "CR.1"
CR_LOWER   = "CR.2"

# Pupil-teacher ratio (trained teachers variants)
PTR_PRIMARY_TRAINED = "PTRHC.1.TRAINED"
PTR_2T3_TRAINED     = "PTRHC.2T3.TRAINED"
PTR_LOWER_TRAINED   = "PTRHC.2.TRAINED"
PTR_UPPER_TRAINED   = "PTRHC.3.TRAINED"

# Read only what we may use (to keep it lean)
needed_indicators = [
    OOS_PRIMARY, OOS_LOWER, OOS_UPPER, OOS_2T3,
    OOS_PRIMARY_F, OOS_PRIMARY_M, OOS_LOWER_F, OOS_LOWER_M, OOS_UPPER_F, OOS_UPPER_M, OOS_2T3_F, OOS_2T3_M,
    CR_PRIMARY, CR_LOWER,
    PTR_PRIMARY_TRAINED, PTR_2T3_TRAINED, PTR_LOWER_TRAINED, PTR_UPPER_TRAINED
]

# -----------------------------
# Helper functions
# -----------------------------
def clamp_0_100(x):
    """Clamp value to [0, 100] to avoid tiny glitches. Keep NaN."""
    if pd.isna(x):
        return np.nan
    x = float(x)
    return max(0.0, min(100.0, x))

def inschool_from_oos(oos):
    """In-school proxy (%) = 100 - out-of-school (%)."""
    if pd.isna(oos):
        return np.nan
    oos = clamp_0_100(oos)
    return 100.0 - oos if not pd.isna(oos) else np.nan

def pick_secondary_oos(row):
    """
    Choose secondary out-of-school:
      1) ROFST.2T3.CP
      2) avg(ROFST.2.CP, ROFST.3.CP) if both
      3) only 2 or only 3
    Returns (value, flag)
    """
    v_2t3 = row.get(OOS_2T3, np.nan)
    if not pd.isna(v_2t3):
        return v_2t3, "2T3"

    v2 = row.get(OOS_LOWER, np.nan)
    v3 = row.get(OOS_UPPER, np.nan)

    if not pd.isna(v2) and not pd.isna(v3):
        return (float(v2) + float(v3)) / 2.0, "avg(2,3)"
    if not pd.isna(v2):
        return v2, "only2"
    if not pd.isna(v3):
        return v3, "only3"

    return np.nan, "missing"

def pick_secondary_oos_by_sex(row, sex):
    """
    Same logic as pick_secondary_oos but for sex-specific indicators.
    sex: "F" or "M"
    """
    if sex == "F":
        v_2t3 = row.get(OOS_2T3_F, np.nan)
        v2 = row.get(OOS_LOWER_F, np.nan)
        v3 = row.get(OOS_UPPER_F, np.nan)
    else:
        v_2t3 = row.get(OOS_2T3_M, np.nan)
        v2 = row.get(OOS_LOWER_M, np.nan)
        v3 = row.get(OOS_UPPER_M, np.nan)

    if not pd.isna(v_2t3):
        return v_2t3, "2T3"
    if not pd.isna(v2) and not pd.isna(v3):
        return (float(v2) + float(v3)) / 2.0, "avg(2,3)"
    if not pd.isna(v2):
        return v2, "only2"
    if not pd.isna(v3):
        return v3, "only3"
    return np.nan, "missing"

def build_wide_row(country_code, country_name, indicator_code, indicator_name, year_to_value):
    """Create one WDI-like wide row."""
    row = {
        "Country Name": country_name,
        "Country Code": country_code,
        "Indicator Name": indicator_name,
        "Indicator Code": indicator_code,
    }
    for y in years_to_keep:
        row[str(y)] = year_to_value.get(y, np.nan)
    return row

try:
    # -----------------------------
    # Load country names
    # -----------------------------
    countries = pd.read_csv(COUNTRY_IN)
    name_map = dict(zip(countries["COUNTRY_ID"], countries["COUNTRY_NAME_EN"]))

    # -----------------------------
    # Load SDG4 national data (filtered)
    # -----------------------------
    sdg = pd.read_csv(SDG_IN, usecols=["INDICATOR_ID", "COUNTRY_ID", "YEAR", "VALUE"], low_memory=False)
    print(f"   > Original SDG shape: {sdg.shape}")

    # Basic filtering: countries + years + indicators
    sdg = sdg[sdg["COUNTRY_ID"].isin(target_countries)]
    sdg = sdg[sdg["YEAR"].isin(years_to_keep)]
    sdg = sdg[sdg["INDICATOR_ID"].isin(needed_indicators)]

    print(f"   > Filtered SDG shape: {sdg.shape}")

    # Convert to numeric (keep NaN if missing)
    sdg["VALUE"] = pd.to_numeric(sdg["VALUE"], errors="coerce")

    # Pivot to (country, year) rows with indicator columns
    tab = sdg.pivot_table(
        index=["COUNTRY_ID", "YEAR"],
        columns="INDICATOR_ID",
        values="VALUE",
        aggfunc="mean"
    ).reset_index()

    # -----------------------------
    # Build output rows
    # -----------------------------
    rows_out = []

    for c in target_countries:
        c_name = name_map.get(c, c)
        df_c = tab[tab["COUNTRY_ID"] == c].copy()
        if df_c.empty:
            continue

        # Map: year -> dict of indicator values
        by_year = {int(r["YEAR"]): r.to_dict() for _, r in df_c.iterrows()}

        # A) Website dumbbell indicators (keep codes main.js expects)
        prim_in = {}
        sec_in  = {}
        sec_src = {}

        for y in years_to_keep:
            rd = by_year.get(y, {})
            prim_in[y] = inschool_from_oos(rd.get(OOS_PRIMARY, np.nan))

            sec_oos, flag = pick_secondary_oos(rd)
            sec_in[y] = inschool_from_oos(sec_oos)
            sec_src[y] = flag

        rows_out.append(build_wide_row(
            c, c_name,
            "SE.PRM.TENR",
            "In-school proxy (100 - Out-of-school), primary age (%)",
            prim_in
        ))

        rows_out.append(build_wide_row(
            c, c_name,
            "SE.SEC.NENR",
            "In-school proxy (100 - Out-of-school), secondary age BEST (%)",
            sec_in
        ))

        # Quality/source flag (so our fallback is transparent)
        rows_out.append(build_wide_row(
            c, c_name,
            "UIS.SEC.SOURCE",
            "Secondary estimate source (2T3 / avg(2,3) / only2 / only3 / missing)",
            sec_src
        ))

        # B) Extra indicators for new graph ideas (raw OOS)
        oos_p = {}
        oos_l = {}
        oos_u = {}
        oos_best = {}

        for y in years_to_keep:
            rd = by_year.get(y, {})
            oos_p[y] = rd.get(OOS_PRIMARY, np.nan)
            oos_l[y] = rd.get(OOS_LOWER, np.nan)
            oos_u[y] = rd.get(OOS_UPPER, np.nan)
            oos_best[y] = pick_secondary_oos(rd)[0]

        rows_out.append(build_wide_row(c, c_name, "UIS.ROFST.1.CP", "Out-of-school rate, primary age (%)", oos_p))
        rows_out.append(build_wide_row(c, c_name, "UIS.ROFST.2.CP", "Out-of-school rate, lower secondary age (%)", oos_l))
        rows_out.append(build_wide_row(c, c_name, "UIS.ROFST.3.CP", "Out-of-school rate, upper secondary age (%)", oos_u))
        rows_out.append(build_wide_row(c, c_name, "UIS.ROFST.SEC.BEST", "Out-of-school rate, secondary age BEST (%)", oos_best))

        # C) Gender gap (in-school proxy by sex)
        ins_p_f, ins_p_m, ins_s_f, ins_s_m = {}, {}, {}, {}
        for y in years_to_keep:
            rd = by_year.get(y, {})

            ins_p_f[y] = inschool_from_oos(rd.get(OOS_PRIMARY_F, np.nan))
            ins_p_m[y] = inschool_from_oos(rd.get(OOS_PRIMARY_M, np.nan))

            sec_f, _ = pick_secondary_oos_by_sex(rd, "F")
            sec_m, _ = pick_secondary_oos_by_sex(rd, "M")

            ins_s_f[y] = inschool_from_oos(sec_f)
            ins_s_m[y] = inschool_from_oos(sec_m)

        rows_out.append(build_wide_row(c, c_name, "UIS.INS.PRM.F", "In-school proxy, primary age, female (%)", ins_p_f))
        rows_out.append(build_wide_row(c, c_name, "UIS.INS.PRM.M", "In-school proxy, primary age, male (%)", ins_p_m))
        rows_out.append(build_wide_row(c, c_name, "UIS.INS.SEC.F", "In-school proxy, secondary age BEST, female (%)", ins_s_f))
        rows_out.append(build_wide_row(c, c_name, "UIS.INS.SEC.M", "In-school proxy, secondary age BEST, male (%)", ins_s_m))

        # D) Completion rates (leakage ideas)
        cr1, cr2 = {}, {}
        for y in years_to_keep:
            rd = by_year.get(y, {})
            cr1[y] = rd.get(CR_PRIMARY, np.nan)
            cr2[y] = rd.get(CR_LOWER, np.nan)

        rows_out.append(build_wide_row(c, c_name, "UIS.CR.1", "Completion rate, primary (%)", cr1))
        rows_out.append(build_wide_row(c, c_name, "UIS.CR.2", "Completion rate, lower secondary (%)", cr2))

        # E) Pupil-teacher ratio (capacity stress ideas)
        ptr_p, ptr_s = {}, {}
        for y in years_to_keep:
            rd = by_year.get(y, {})
            ptr_p[y] = rd.get(PTR_PRIMARY_TRAINED, np.nan)

            # Secondary PTR fallback: prefer 2T3, else avg(2,3), else 2/3
            v_2t3 = rd.get(PTR_2T3_TRAINED, np.nan)
            v2 = rd.get(PTR_LOWER_TRAINED, np.nan)
            v3 = rd.get(PTR_UPPER_TRAINED, np.nan)

            if not pd.isna(v_2t3):
                ptr_s[y] = v_2t3
            elif not pd.isna(v2) and not pd.isna(v3):
                ptr_s[y] = (float(v2) + float(v3)) / 2.0
            elif not pd.isna(v2):
                ptr_s[y] = v2
            elif not pd.isna(v3):
                ptr_s[y] = v3
            else:
                ptr_s[y] = np.nan

        rows_out.append(build_wide_row(c, c_name, "UIS.PTR.PRM.TRAINED", "Pupil-teacher ratio, primary (trained) (ratio)", ptr_p))
        rows_out.append(build_wide_row(c, c_name, "UIS.PTR.SEC.TRAINED", "Pupil-teacher ratio, secondary BEST (trained) (ratio)", ptr_s))

    edu_clean = pd.DataFrame(rows_out)

    # Drop rows with ALL missing across selected years (numeric only)
    # Note: UIS.SEC.SOURCE is text flags, so we keep it even if missing years exist.
    numeric_like = edu_clean[years_str].apply(pd.to_numeric, errors="coerce")
    keep_mask = ~numeric_like.isna().all(axis=1) | (edu_clean["Indicator Code"] == "UIS.SEC.SOURCE")
    edu_clean = edu_clean[keep_mask].copy()

    # Sort for readability
    edu_clean = edu_clean.sort_values(["Country Code", "Indicator Code"]).reset_index(drop=True)

    edu_clean.to_csv(EDU_OUT, index=False)

    print(f"✅ UNESCO Education Done! Saved: {EDU_OUT} | shape={edu_clean.shape}")
    print(f"   > Countries kept: {sorted(edu_clean['Country Code'].unique().tolist())}")
    print(f"   > Indicators kept: {sorted(edu_clean['Indicator Code'].unique().tolist())}")

    # Quick Uganda check (because it always causes drama)
    uga = edu_clean[edu_clean["Country Code"] == "UGA"][["Indicator Code"] + years_str]
    if not uga.empty:
        print("\n🔎 Uganda check (UGA):")
        print(uga.head(12))

except FileNotFoundError as e:
    print(f"❌ File not found: {e}")
except Exception as e:
    print(f"❌ Error while cleaning UNESCO education data: {e}")



🚀 Starting UNESCO SDG4 education cleaning...
   > Original SDG shape: (1372669, 4)
   > Filtered SDG shape: (1244, 4)
✅ UNESCO Education Done! Saved: ./data/clean_education.csv | shape=(157, 19)
   > Countries kept: ['AFG', 'DEU', 'ETH', 'IRN', 'JOR', 'KEN', 'LBN', 'PAK', 'SDN', 'SYR', 'TUR', 'UGA']
   > Indicators kept: ['SE.PRM.TENR', 'SE.SEC.NENR', 'UIS.CR.1', 'UIS.CR.2', 'UIS.INS.PRM.F', 'UIS.INS.PRM.M', 'UIS.INS.SEC.F', 'UIS.INS.SEC.M', 'UIS.PTR.PRM.TRAINED', 'UIS.PTR.SEC.TRAINED', 'UIS.ROFST.1.CP', 'UIS.ROFST.2.CP', 'UIS.ROFST.3.CP', 'UIS.ROFST.SEC.BEST', 'UIS.SEC.SOURCE']

🔎 Uganda check (UGA):
          Indicator Code      2010       2011     2012      2013     2014  \
149          SE.PRM.TENR  91.11917   94.40354      NaN  95.88343      NaN   
150             UIS.CR.1       NaN  39.383381      NaN       NaN      NaN   
151             UIS.CR.2       NaN  23.135651      NaN       NaN      NaN   
152        UIS.INS.PRM.F  91.77857   94.98692      NaN  96.57951      NaN   
153  

In [14]:
# =========================================================
# UNESCO SDG4 EDUCATION CLEANING (UIS SDG4 Bulk Data)
#
# Inputs:
#   - SDG_DATA_NATIONAL.csv    (long format: indicator-country-year-value)
#   - SDG_COUNTRY.csv          (country names)
#   - clean_demographics.csv   (to derive asylum countries from your project)
#
# Output:
#   - clean_education.csv (wide format like WDI: year columns)
#
# Key design decisions:
# 1) Countries:
#    - Derived automatically from demographics:
#      all Country of Asylum codes appearing in demographics,
#      excluding rows where Origin == Asylum (self-flow).
#    - Optional: keep WLD if present in SDG_COUNTRY.
#
# 2) Indicators (2010-2024):
#    - Primary OOS: ROFST.1.CP
#    - Secondary OOS BEST (STRICT): ROFST.2T3.CP only (NO fallback to 2/3)
#    - Enrollment proxy (%): 100 - OOS  (clamped to [0,100])
#    - Gender OOS (optional): ROFST.1.F/M.CP and ROFST.2T3.F/M.CP
#    - PTR trained: PTRHC.1.TRAINED and PTRHC.2T3.TRAINED
#      + Transparent fallback for secondary PTR (avg(2,3) or only2/only3) with a source flag
#
# Why strict BEST for secondary OOS?
# - Prevents Kenya-style distortions: 2T3 is a distinct official indicator.
#   Lower-only or upper-only is NOT "BEST". If 2T3 is missing => missing.
# =========================================================

import pandas as pd
import numpy as np

print("\n🚀 Starting UNESCO SDG4 education cleaning (asylum-driven, STRICT secondary BEST)...")

# Paths 
SDG_IN = "./data/SDG_DATA_NATIONAL.csv"
COUNTRY_IN = "./data/SDG_COUNTRY.csv"
DEMO_IN = "./data/clean_demographics.csv"
EDU_OUT = "./data/clean_educations.csv"

# Years 2010-2024
years_to_keep = list(range(2010, 2025))
years_str = [str(y) for y in years_to_keep]

# -----------------------------
# UNESCO SDG4 Indicator IDs (raw)
# -----------------------------
# Out-of-school (both sexes)
OOS_PRIMARY = "ROFST.1.CP"
OOS_2T3     = "ROFST.2T3.CP"     # Secondary BEST (STRICT)

# Out-of-school by sex (optional but useful)
OOS_PRIMARY_F = "ROFST.1.F.CP"
OOS_PRIMARY_M = "ROFST.1.M.CP"
OOS_2T3_F     = "ROFST.2T3.F.CP"
OOS_2T3_M     = "ROFST.2T3.M.CP"

# PTR trained (capacity)
PTR_PRIMARY_TRAINED = "PTRHC.1.TRAINED"
PTR_2T3_TRAINED     = "PTRHC.2T3.TRAINED"
# Optional PTR fallback parts (transparent if used)
PTR_LOWER_TRAINED   = "PTRHC.2.TRAINED"
PTR_UPPER_TRAINED   = "PTRHC.3.TRAINED"

needed_indicators = [
    OOS_PRIMARY, OOS_2T3,
    OOS_PRIMARY_F, OOS_PRIMARY_M, OOS_2T3_F, OOS_2T3_M,
    PTR_PRIMARY_TRAINED, PTR_2T3_TRAINED, PTR_LOWER_TRAINED, PTR_UPPER_TRAINED
]

# -----------------------------
# Helper functions
# -----------------------------
def clamp_0_100(x):
    """Clamp numeric values to [0, 100]. Keep NaN as NaN."""
    if pd.isna(x):
        return np.nan
    x = float(x)
    return max(0.0, min(100.0, x))

def inschool_from_oos(oos):
    """In-school proxy (%) = 100 - out-of-school (%)."""
    if pd.isna(oos):
        return np.nan
    oos = clamp_0_100(oos)
    return 100.0 - oos if not pd.isna(oos) else np.nan

def pick_secondary_best_strict(row):
    """
    STRICT secondary BEST:
    - Only accept ROFST.2T3.CP
    - If missing => NaN
    Returns (value, flag)
    """
    v = row.get(OOS_2T3, np.nan)
    if not pd.isna(v):
        return v, "2T3"
    return np.nan, "missing"

def pick_secondary_best_strict_by_sex(row, sex):
    """STRICT by sex: only ROFST.2T3.F/M.CP"""
    v = row.get(OOS_2T3_F if sex == "F" else OOS_2T3_M, np.nan)
    if not pd.isna(v):
        return v, "2T3"
    return np.nan, "missing"

def pick_ptr_secondary(row):
    """
    PTR secondary (trained):
    - Prefer PTRHC.2T3.TRAINED
    - Else transparent fallback:
      avg(PTRHC.2.TRAINED, PTRHC.3.TRAINED) if both
      else only2 or only3
    Returns (value, flag)
    """
    v_2t3 = row.get(PTR_2T3_TRAINED, np.nan)
    if not pd.isna(v_2t3):
        return v_2t3, "2T3"

    v2 = row.get(PTR_LOWER_TRAINED, np.nan)
    v3 = row.get(PTR_UPPER_TRAINED, np.nan)

    if not pd.isna(v2) and not pd.isna(v3):
        return (float(v2) + float(v3)) / 2.0, "avg(2,3)"
    if not pd.isna(v2):
        return v2, "only2"
    if not pd.isna(v3):
        return v3, "only3"

    return np.nan, "missing"

def build_wide_row(country_code, country_name, indicator_code, indicator_name, year_to_value):
    """Create one WDI-like wide row."""
    row = {
        "countryName": country_name,
        "countryCode": country_code,
        "indicatorName": indicator_name,
        "indicatorCode": indicator_code,
    }
    for y in years_to_keep:
        row[str(y)] = year_to_value.get(y, np.nan)
    return row

def detect_col(df, candidates):
    """Pick the first matching column name from candidates."""
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

try:
    # -----------------------------
    # Load country names
    # -----------------------------
    countries = pd.read_csv(COUNTRY_IN)
    # Expected columns in SDG_COUNTRY: COUNTRY_ID, COUNTRY_NAME_EN
    name_map = dict(zip(countries["COUNTRY_ID"], countries["COUNTRY_NAME_EN"]))

    # -----------------------------
    # Derive target countries from demographics (asylum countries)
    # - Exclude self-flow (origin == asylum)
    # -----------------------------
    demo = pd.read_csv(DEMO_IN, low_memory=False)

    asylum_col = detect_col(demo, [
        "Country of Asylum Code", "asylumCode", "asylum_country_code", "AsylumCode"
    ])
    origin_col = detect_col(demo, [
        "Country of Origin Code", "originCode", "origin_country_code", "OriginCode"
    ])

    if asylum_col is None or origin_col is None:
        raise ValueError(
            "Could not detect asylum/origin code columns in demographics. "
            "Expected something like 'Country of Asylum Code' and 'Country of Origin Code'."
        )

    # Keep only rows where asylum != origin (avoid self-flow artifacts)
    demo_valid = demo[(demo[asylum_col].notna()) & (demo[origin_col].notna())].copy()
    demo_valid = demo_valid[demo_valid[asylum_col] != demo_valid[origin_col]]

    target_countries = sorted(demo_valid[asylum_col].unique().tolist())

    # Optional: keep WLD if exists in SDG country list
    if "WLD" in name_map and "WLD" not in target_countries:
        target_countries.append("WLD")

    print(f"   > Derived asylum countries (no self-flow): {len(target_countries)}")

    # -----------------------------
    # Load SDG4 national data (filtered)
    # -----------------------------
    sdg = pd.read_csv(
        SDG_IN,
        usecols=["INDICATOR_ID", "COUNTRY_ID", "YEAR", "VALUE"],
        low_memory=False
    )
    print(f"   > Original SDG shape: {sdg.shape}")

    sdg = sdg[sdg["COUNTRY_ID"].isin(target_countries)]
    sdg = sdg[sdg["YEAR"].isin(years_to_keep)]
    sdg = sdg[sdg["INDICATOR_ID"].isin(needed_indicators)]
    print(f"   > Filtered SDG shape: {sdg.shape}")

    sdg["VALUE"] = pd.to_numeric(sdg["VALUE"], errors="coerce")

    # Pivot to (country, year) with indicator columns
    tab = sdg.pivot_table(
        index=["COUNTRY_ID", "YEAR"],
        columns="INDICATOR_ID",
        values="VALUE",
        aggfunc="mean"
    ).reset_index()

    # -----------------------------
    # Build output rows
    # -----------------------------
    rows_out = []

    for c in target_countries:
        c_name = name_map.get(c, c)

        df_c = tab[tab["COUNTRY_ID"] == c].copy()
        if df_c.empty:
            continue

        by_year = {int(r["YEAR"]): r.to_dict() for _, r in df_c.iterrows()}

        # ---- A) Core indicators for your charts ----
        # Raw OOS
        oos_primary = {}
        oos_secondary_best = {}
        sec_best_src = {}

        # Enrollment proxy (in-school)
        ins_primary = {}
        ins_secondary = {}

        # Gender in-school proxies
        ins_p_f, ins_p_m = {}, {}
        ins_s_f, ins_s_m = {}, {}

        # PTR trained
        ptr_primary = {}
        ptr_secondary = {}
        ptr_secondary_src = {}

        for y in years_to_keep:
            rd = by_year.get(y, {})

            # Primary OOS
            op = rd.get(OOS_PRIMARY, np.nan)
            oos_primary[y] = op
            ins_primary[y] = inschool_from_oos(op)

            # Secondary BEST OOS (STRICT 2T3)
            os, flag = pick_secondary_best_strict(rd)
            oos_secondary_best[y] = os
            sec_best_src[y] = flag
            ins_secondary[y] = inschool_from_oos(os)

            # Gender (primary)
            ins_p_f[y] = inschool_from_oos(rd.get(OOS_PRIMARY_F, np.nan))
            ins_p_m[y] = inschool_from_oos(rd.get(OOS_PRIMARY_M, np.nan))

            # Gender (secondary BEST STRICT)
            sec_f, _ = pick_secondary_best_strict_by_sex(rd, "F")
            sec_m, _ = pick_secondary_best_strict_by_sex(rd, "M")
            ins_s_f[y] = inschool_from_oos(sec_f)
            ins_s_m[y] = inschool_from_oos(sec_m)

            # PTR trained
            ptr_primary[y] = rd.get(PTR_PRIMARY_TRAINED, np.nan)
            ps, pflag = pick_ptr_secondary(rd)
            ptr_secondary[y] = ps
            ptr_secondary_src[y] = pflag

        # --- Output rows (codes aligned with your JS expectations) ---
        # Enrollment proxies (used by dumbbell/scatter)
        rows_out.append(build_wide_row(
            c, c_name,
            "SE.PRM.TENR",
            "In-school proxy (100 - ROFST.1.CP), primary age (%)",
            ins_primary
        ))
        rows_out.append(build_wide_row(
            c, c_name,
            "SE.SEC.NENR",
            "In-school proxy (100 - ROFST.2T3.CP), secondary age BEST (%) [STRICT]",
            ins_secondary
        ))

        # Raw OOS for Cliff (use these directly)
        rows_out.append(build_wide_row(
            c, c_name,
            "UIS.ROFST.1.CP",
            "Out-of-school rate, primary age (%) (ROFST.1.CP)",
            oos_primary
        ))
        rows_out.append(build_wide_row(
            c, c_name,
            "UIS.ROFST.SEC.BEST",
            "Out-of-school rate, secondary age BEST (%) (ROFST.2T3.CP) [STRICT]",
            oos_secondary_best
        ))
        rows_out.append(build_wide_row(
            c, c_name,
            "UIS.SEC.SOURCE",
            "Secondary BEST source flag (2T3 / missing) [STRICT]",
            sec_best_src
        ))

        # Gender proxies (nice for future charts)
        rows_out.append(build_wide_row(c, c_name, "UIS.INS.PRM.F", "In-school proxy, primary age, female (%)", ins_p_f))
        rows_out.append(build_wide_row(c, c_name, "UIS.INS.PRM.M", "In-school proxy, primary age, male (%)", ins_p_m))
        rows_out.append(build_wide_row(c, c_name, "UIS.INS.SEC.F", "In-school proxy, secondary BEST (2T3), female (%) [STRICT]", ins_s_f))
        rows_out.append(build_wide_row(c, c_name, "UIS.INS.SEC.M", "In-school proxy, secondary BEST (2T3), male (%) [STRICT]", ins_s_m))

        # PTR trained
        rows_out.append(build_wide_row(c, c_name, "UIS.PTR.PRM.TRAINED", "Pupil-teacher ratio, primary (trained) (ratio)", ptr_primary))
        rows_out.append(build_wide_row(c, c_name, "UIS.PTR.SEC.TRAINED", "Pupil-teacher ratio, secondary (trained) BEST (ratio)", ptr_secondary))
        rows_out.append(build_wide_row(c, c_name, "UIS.PTR.SEC.SOURCE", "PTR secondary source flag (2T3 / avg(2,3) / only2 / only3 / missing)", ptr_secondary_src))

    edu_clean = pd.DataFrame(rows_out)

    # Drop rows with ALL missing across numeric years
    # Keep SOURCE flags even if many missing (they are informative)
    numeric_like = edu_clean[years_str].apply(pd.to_numeric, errors="coerce")
    keep_mask = ~numeric_like.isna().all(axis=1) | edu_clean["indicatorCode"].isin(["UIS.SEC.SOURCE", "UIS.PTR.SEC.SOURCE"])
    edu_clean = edu_clean[keep_mask].copy()

    # Sort for readability
    edu_clean = edu_clean.sort_values(["countryCode", "indicatorCode"]).reset_index(drop=True)

    edu_clean.to_csv(EDU_OUT, index=False)

    print(f"\n✅ Education cleaning done. Saved: {EDU_OUT} | shape={edu_clean.shape}")
    print(f"   > Countries kept: {edu_clean['countryCode'].nunique()}")
    print(f"   > Indicators kept: {sorted(edu_clean['indicatorCode'].unique().tolist())}")

    # Quick sanity: show how many countries have STRICT secondary BEST at least once
    sec_rows = edu_clean[edu_clean["indicatorCode"] == "UIS.ROFST.SEC.BEST"].copy()
    if not sec_rows.empty:
        sec_numeric = sec_rows[years_str].apply(pd.to_numeric, errors="coerce")
        ok_any = (~sec_numeric.isna()).any(axis=1)
        print(f"   > Countries with any secondary BEST (2T3) available: {int(ok_any.sum())} / {len(sec_rows)}")

except FileNotFoundError as e:
    print(f"❌ File not found: {e}")
except Exception as e:
    print(f"❌ Error while cleaning UNESCO education data: {e}")



🚀 Starting UNESCO SDG4 education cleaning (asylum-driven, STRICT secondary BEST)...
   > Derived asylum countries (no self-flow): 191
   > Original SDG shape: (1372669, 4)
   > Filtered SDG shape: (13059, 4)

✅ Education cleaning done. Saved: ./data/clean_educations.csv | shape=(2028, 19)
   > Countries kept: 185
   > Indicators kept: ['SE.PRM.TENR', 'SE.SEC.NENR', 'UIS.INS.PRM.F', 'UIS.INS.PRM.M', 'UIS.INS.SEC.F', 'UIS.INS.SEC.M', 'UIS.PTR.PRM.TRAINED', 'UIS.PTR.SEC.SOURCE', 'UIS.PTR.SEC.TRAINED', 'UIS.ROFST.1.CP', 'UIS.ROFST.SEC.BEST', 'UIS.SEC.SOURCE']
   > Countries with any secondary BEST (2T3) available: 173 / 173
